In [ ]:
import os
import sys

IS_COLAB = "google.colab" in sys.modules
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_COLAB:
    print("Colab environment. Make sure the HW_DIR below is matches what you used in the startup notebook.")
    print("If you are asked to restart the runtime, you do not need to do so.")
    print("If you decide to restart the runtime, you will need to re-run this cell.")
    from google.colab import drive
    drive.mount("/content/drive")
    HW_DIR = "/content/drive/MyDrive/CS189/hw/hw1-pt1"
    os.chdir(HW_DIR)
    %pip install -r colab_requirements.txt
elif IS_DATABRICKS:
    print("Databricks environment. Make sure your base environment is ML v5.")
    %pip install -r databricks_requirements.txt
else:
    print("Local development. Make sure your kernel has everything in requirements.txt.")


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("fashion_pt_1.ipynb")

<link rel="stylesheet" href="berkeley.css">

<h1 class="cal cal-h1">Homework 1.1 – AGI, Everywhere, All at Once</h1>

Welcome to Homework 1.1! In this assignment, you will practice core numerical tools (`numpy`, `pytorch` autograd), explore Fashion-MNIST with a thin `pandas` metadata table, train a simple classifier, and implement image transforms as matrices.

---

## Due Date: Friday, September 18, 11:59 PM

This assignment is due on **Friday, September 18, at 11:59 PM**. You must submit your work to the programming assignment by this deadline. Please refer to the syllabus for the [Slip Day policy](https://eecs189.org/fa26/syllabus/#slip-days). No late submissions will be accepted beyond the details outlined in the Slip Day policy.

### Submission Tips:
- **Plan ahead**: We strongly encourage you to submit your work several hours before the deadline. This will give you ample time to address any submission issues.
- **Reach out for help early**: If you encounter difficulties, contact course staff well before the deadline. While we are happy to assist with submission issues, we cannot guarantee responses to last-minute requests.

---

## Assignment Overview

This notebook contains a series of tasks designed to help you refresh math pre-requisites and practice key concepts in data manipulation and visualization. You will implement all coding TODOs in the notebook and respond to written questions separately in a pdf. Some tasks are open-ended, which allows you to explore and experiment with different approaches.

### Key Learning Objectives:
1. Work with `numpy` and `pytorch` for linear algebra and calculus.
2. Visualize data using `plotly` and `pandas`' built-in plotting functions.
3. Gain experience with organizing and analyzing datasets.
4. Understand the importance of data exploration and preprocessing.

---

<div align="center">

### Grading Breakdown

| Question | Manual Grading? | Points |
|----------|-----------------|--------|
| 1a       | No              | 2      |
| 1b       | No              | 2      |
| 1c       | No              | 2      |
| 2a       | No              | 1      |
| 2b       | No              | 2      |
| 2ci      | No              | 0.5    |
| 2cii     | Yes             | 0.5    |
| 3a       | No              | 2      |
| 3b       | Yes             | 1      |
| 3c       | Yes             | 1      |
| 3d       | Yes             | 2      |
| 4a       | No              | 1      |
| 4b       | No              | 2      |
| 4ci      | No              | 0.5    |
| 4cii     | Yes             | 0.5    |
| 4d       | Yes             | 2      |
| 4ei      | No              | 1.5    |
| 4eii     | Yes             | 0.5    |
| 4fi      | No              | 0.5    |
| 4fii     | Yes             | 0.5    |
| 4gi      | No              | 0.5    |
| 4gii     | Yes             | 0.5    |
| 4h       | Yes             | 1      |
| 5a       | No              | 1      |
| 5b       | No              | 2      |
| 5c       | No              | 2      |
| 5d       | No              | 2      |
| 5e       | No              | 2      |
| 5f       | No              | 1      |
| 5g       | Yes             | 2      |
| 5hi      | No              | 0.75   |
| 5hii     | Yes             | 0.25   |
| 5i       | No              | 2      |
| 5j       | Yes             | 2      |
| **Total**|                 | **44** |

</div>

**Note**: "Manual" questions are written responses saved under `written/` and uploaded as a PDF to the **associated paper assignment**. Coding questions are graded automatically.

---

### Instructions:
1. Carefully read each question and its requirements.
2. Complete all TODOs in the notebook. You may add extra lines of code if needed.
3. For manual questions, provide clear and concise written responses in a separate document.
4. Test your code thoroughly to ensure it meets the requirements.


---

### Autograding
- After each coding question, run `grader.check("q...")` to execute the **public** tests.
- Keep the `tests/` folder next to this notebook. If staff announces a test update, use the cell below to restore only `tests/` from the announced tag.
- The autograder on Pensive runs additional tests that are not in this handout. You will see pass/fail and error messages for those tests when you submit.
- Your responses to manually graded questions (e.g., figures, text responses) are saved into a `written` folder.
- Submit your coding `.ipynb`, any required model files, and manually graded files folder to the **programming** assignment. Your manually graded files **must** be under a `written` folder in your submission.
- The export cells at the end of the notebook packages everything into a `.zip` you can submit and will warn you of any missing files. This is the preferred way to submit your work.
- A PDF of your manually graded responses will be automatically uploaded to the associated pdf assignment. Please verify there were no issues with the upload.

Good luck!


In [ ]:
TESTS_VERSION = "v1"
REPULL_TESTS = False

if REPULL_TESTS:
    !git fetch origin tag {TESTS_VERSION}
    !git restore --source={TESTS_VERSION} -- tests/


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torchvision
import os
import random
from IPython.display import display
import joblib
from sklearn.cluster import KMeans
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
import plotly.io as pio

pd.options.plotting.backend = "plotly" # set plotting backend to plotly
pio.renderers.default = "plotly_mimetype+png"  # PNG so the export PDF can embed figures

### **IMPORTANT:**
- Do not change the random seed values!!!
- Before you submit your notebook, remember to set `save_models=True` and `load_saved_models=True`. This saves your final models which we will use for the autograder. Set these to false if you are still tweaking your model setup. We have provided code for saving models - **do not change these file names!!**
- When uploading your notebook, include `classifier.joblib` and the `written/` folder (figures and written answers)


In [ ]:
# Set random seeds for reproducible results
SEED = 189
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# IMPORTANT: set save_models to True to save trained models. YOU NEED TO DO THIS FOR THE AUTOGRADER TO WORK.
save_models = True
load_saved_models = True  # After training, you can set this to True to load the saved models and not have to re-train them.
IS_GRADING_ENV = os.getenv("IS_GRADING_ENV") == "true"
if IS_GRADING_ENV:
    pio.renderers.default = "json"  # no kaleido PNG
    load_saved_models = True


In [ ]:
def show_images(images, max_images=40, ncols=5, labels=None, reshape=False):
    '''Visualize a batch of images with shape [N, H, W] (or flattened if reshape=True).'''
    if isinstance(images, list):
        images = np.stack(images)
    images = np.asarray(images)
    if reshape:
        images = images.reshape(images.shape[0], 28, 28)
    # Limit the number of images to display
    n = min(images.shape[0], max_images)
    px_height = 220
    fig = px.imshow(
        images[:n],
        color_continuous_scale="gray_r",
        facet_col=0,
        facet_col_wrap=ncols,
        height=px_height * int(np.ceil(n / ncols)),
    )
    fig.update_layout(coloraxis_showscale=False)
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False)
    if labels is not None:
        # Extract the facet number and replace with the label.
        fig.for_each_annotation(lambda a: a.update(text=labels[int(a.text.split("=")[-1])]))
    return fig


In [ ]:
import json

WRITTEN_DIR = "written"
os.makedirs(WRITTEN_DIR, exist_ok=True)

# Shipped in tests/ by build_release.py from PARTS[n]["written_files"].
with open(os.path.join("tests", "written_files.json"), encoding="utf-8") as f:
    EXPECTED_WRITTEN_FILES = json.load(f)


def save_figure(fig, question_name):
    """Save a Plotly figure as written/{question_name}.png for the written submission."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.png")
    try:
        fig.write_image(path)
    except Exception as e:
        print(f"WARNING: Could not save {path}: {e}")
        return None
    print(f"Saved {path}")
    return path


def save_written_answer(answer, question_name):
    """Save a written response as written/{question_name}.txt for the written submission."""
    if IS_GRADING_ENV:
        return None
    path = os.path.join(WRITTEN_DIR, f"{question_name}.txt")
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(str(answer).strip() + "\n")
    except Exception as e:
        print(f"WARNING: Could not save {path}: {e}")
        return None
    print(f"Saved {path}")
    return path


def warn_missing_written_files():
    """Print which expected written/ files are missing locally. Does not fail the cell."""
    missing = [
        name
        for name in EXPECTED_WRITTEN_FILES
        if not os.path.isfile(os.path.join(WRITTEN_DIR, name))
    ]
    if missing:
        print("WARNING: Missing expected files in written/:")
        for name in missing:
            print(f"  - {name}")
        print("Re-run the cells that save these artifacts before exporting.")
        return missing
    print(f"All {len(EXPECTED_WRITTEN_FILES)} expected files found in written/.")
    return []


## Problem 1: Linear Algebra with NumPy and Gradients with PyTorch

This section scales up problems from the written homework: eigenvalues and matrix powers with **NumPy**, maximizing $\|Ax\|$ via the SVD, and partial derivatives with **PyTorch** autograd.

PyTorch also exposes linear-algebra routines under `torch.linalg` (e.g. `torch.linalg.eig`, `torch.linalg.svd`) with APIs similar to NumPy. Here we use NumPy for the matrix problems and reserve PyTorch for automatic differentiation.


### Problem 1a: Asymptotic powers of matrices

This is a generalized version of written Problem 2. Implement

`matrix_power_to_zero(M) -> bool`

that takes an arbitrary square $N \times N$ matrix $M$ and returns `True` if $M^n \to \mathbf{0}$ as $n \to \infty$ and `False` otherwise. Here $\mathbf{0}$ denotes the all-zeros matrix of the same shape as $M$. See the written homework for hints on using the eigen-decomposition.


In [ ]:
def matrix_power_to_zero(M):
    """Return True if M^n -> 0 as n -> inf, else False."""
    ...


In [ ]:
grader.check("q1a")

### Problem 1b: Maximizing $\|Ax\|$ via the Singular Value Decomposition

This is a generalized version of written Problem 3. Implement `maximize_Ax(A)` that returns `x_star` where `x_star` is a `numpy` array representing a unit vector maximizing $\|A x\|$ over $\|x\|=1$. See the written homework for hints on the SVD.


In [ ]:
def maximize_Ax(A):
    """Return a unit vector x_star maximizing ||A x||."""
    ...


In [ ]:
grader.check("q1b")

### Problem 1c: Autograd for a scalar function

Let $f(x, y) = e^{xy} + x^2 y$ (as on the written homework). Implement `autograd_f_grads(x_val, y_val)` that uses PyTorch autograd to return `(grad_x, grad_y)` as Python floats at the given point.


In [ ]:
def autograd_f_grads(x_val, y_val):
    """Return (df/dx, df/dy) at (x_val, y_val) for f = exp(xy) + x^2 y."""
    ...


In [ ]:
grader.check("q1c")

<link rel="stylesheet" href="berkeley.css">

## Problem 2: Fashion-MNIST

In this homework, we work with the Fashion-MNIST dataset, consisting of 70k (60k training, 10k testing) grayscale $28\times 28$ images of clothing across 10 classes:
1. **T-shirt/top**
2. **Trouser**
3. **Pullover**
4. **Dress**
5. **Coat**
6. **Sandal**
7. **Shirt**
8. **Sneaker**
9. **Bag**
10. **Ankle boot**

>[Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms.](https://arxiv.org/abs/1708.07747) Han Xiao, Kashif Rasul, Roland Vollgraf.
> https://github.com/zalandoresearch/fashion-mnist

We load the data with [torchvision](https://docs.pytorch.org/vision/stable/index.html) and store pixels as a NumPy array `X_all` of shape `(N, 784)`. A pandas DataFrame `meta` links the images and labels, storing the original Fashion-MNIST index and label.


In [ ]:
# Load the FashionMNIST dataset from torchvision
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)

# Pixels as a dense NumPy array; labels as strings
X_all = train_data.data.numpy().astype(float).reshape(-1, 784)
targets = train_data.targets.numpy()
class_dict = {i: class_name for i, class_name in enumerate(train_data.classes)}
y_all = np.array([class_dict[t] for t in targets])
class_names = list(class_dict.values())

# Thin metadata table: index into X_all / original FMNIST row, plus label
meta = pd.DataFrame({
    "fmnist_idx": np.arange(len(X_all)),
    "label": y_all,
})

print("Loaded FashionMNIST dataset with {} samples.".format(len(X_all)))
print("Classes: {}".format(class_dict))
print("X_all shape:", X_all.shape)
display(meta.head())


### Why not store images in `pandas`?

Most numerical libraries (`numpy`, `sklearn`, `torch`) expect contiguous array-compatible inputs. Storing each image as a Python object inside a DataFrame column would require us to cast the column into an array every time.


### Problem 2a: Label distribution

**Task:** Using `meta`, compute `label_distribution` with [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) and set `is_balanced` to whether every class has the same count.


In [ ]:
# TODO: Calculate the distribution of labels using value_counts()
...

print("Label distribution:")
print(label_distribution)
print("\nIs the dataset balanced?", is_balanced)


In [ ]:
grader.check("q2a")

### Problem 2b: Mean intensity per class

**Task:** Produce a dictionary `mean_brightness` mapping each class label (string) to that class's mean image brightness (mean pixel intensity across all pixels and all images of that class).


In [ ]:
# TODO: create mean_brightness: dict[str, float] mapping label -> mean brightness
...

print(mean_brightness)


In [ ]:
grader.check("q2b")

### Problem 2c: Visualizing class examples

**Task:** For each class in `class_names`, select 2 images from `X_all`. Store the stacked images in `sample_imgs` with shape `(20, 784)` and the corresponding labels in `sample_labels` (length 20).

You will plot these images in the next part.


In [ ]:
# TODO: Get 2 sample images per class. Store them in `sample_imgs` and `sample_labels`.
...


In [ ]:
grader.check("q2ci")

<!-- BEGIN QUESTION -->

**Task:** Visualize `sample_imgs` with `show_images`. Use `sample_labels` as the labels.


In [ ]:
# TODO: Plot `sample_imgs` with `show_images`. Assign the figure to `fig_q2cii`.
...
if not IS_GRADING_ENV:
    fig_q2cii.show()
save_figure(fig_q2cii, "q2cii")


<!-- END QUESTION -->

## Problem 3: Understanding Data Structure with Clustering

Before training classifiers, we explore the data structure using **k-means clustering**, an unsupervised learning method. This helps identify patterns and relationships in the dataset.

**Why Clustering?**

- **Discover Similarities**: Group similar clothing items based on pixel values.
- **Data Insights**: Understand dataset structure to guide modeling.
- **Simplify Data**: Potential preprocessing or dimensionality reduction.

**Steps:**
1. Sample a subset of flattened images from `X_all`.
2. Apply k-means to group images.
3. Analyze clusters for patterns.

Images are already flattened in `X_all` with shape `(N, 784)`.


### Problem 3a: K-means Clustering on the Pixels

Use K-means clustering to group similar images based on their pixel values. This will help us understand how well the algorithm can identify patterns in the dataset without using the labels.

**Task**:
1. Use the [sklearn's `KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) class to cluster the images into `10` clusters (since there are 10 classes in the dataset). For efficiency we will only cluster a 1000 image sample (`meta_sample`).
2. Create a `DataFrame` called `kmeans_df` with the following columns:
    - `fmnist_idx`: the Fashion-MNIST index of the image (in `X_all`).
    - `label`: the true class label of the image.
    - `cluster`: the cluster label assigned by K-means.

**Instructions**:
- When clustering, set `random_state=SEED` for reproducibility.

**Expected Output**:
The `kmeans_df` DataFrame should look like this:

| cluster | label       | fmnist_idx |
|---------|-------------|------------|
| 7       | Ankle boot  | 1523       |
| 6       | T-shirt/top | 8841       |


In [ ]:
# TODO: Perform k-means clustering on the images (10 clusters to match the number of classes)

meta_sample = meta.sample(n=1000, random_state=SEED)
...

kmeans_df.head(3)


In [ ]:
grader.check("q3a")

<!-- BEGIN QUESTION -->

### Problem 3b: Evaluating K-means Clustering

K-means clustering groups data points into clusters based on their similarity. To evaluate how well the clustering algorithm has separated the classes, we can analyze the distribution of true labels within each cluster.

**Task:** 
1. Use the `kmeans_df` `DataFrame` to calculate the distribution of true labels (`label`) within each cluster (`cluster`).
2. Create a [stacked bar plot](https://plotly.com/python/bar-charts/) to visualize the label counts per cluster. Each bar should represent a cluster, and the segments of the bar should represent the counts of each label within that cluster.

**Hint:** If you are running into issues where there are bars “hidden” behind other ones in your Plotly bar chart, try making sure you use fillna(0) or unstack(fill_value=0) after grouping by your KMean clusters.

In [ ]:
# TODO: Create a stacked bar plot of the label counts per cluster. Assign it to `fig_q3b`.
...
if not IS_GRADING_ENV:
    fig_q3b.show()
save_figure(fig_q3b, "q3b")


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Problem 3c: Visualizing Clusters

To better understand the clusters formed by the K-means algorithm, we will visualize a few sample images from each cluster. This will help us identify patterns or similarities among images within the same cluster.

**Task:**
1. For each cluster, randomly sample 7 images.
2. Use the `show_images` function to display the sampled images in a grid.
3. Observe the visual similarities among images in the same cluster.

In [ ]:
# TODO: Plot 7 images from each cluster (use the show_images function, 10 rows, 7 columns)
...
if not IS_GRADING_ENV:
    fig_q3c.show()
save_figure(fig_q3c, "q3c")


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Problem 3d: Observing Patterns in K-means Clustering

Reflecting on the visualizations from the previous part, we observe that the k-means clustering algorithm groups images not only by their clothing category (class) but also by other shared characteristics. 

**Question:** Besides the clothing category, what other visual or structural characteristics of the images might the k-means clustering algorithm be grouping together?

Write your answer in the code cell below (in the `q3d_answer` string) and run that cell to save it as `written/q3d.txt`.


In [ ]:
# TODO: Write your answer in `q3d_answer`, then run this cell to save it.
q3d_answer = """
...
"""
save_written_answer(q3d_answer, "q3d")


<!-- END QUESTION -->

## Problem 4: Training a Classifier

In this section, we will train a machine learning classifier to predict clothing categories from image pixel data. Specifically, we will use a Multi-Layer Perceptron (MLP) classifier, which is a type of neural network.

### Workflow Overview

We will follow a structured workflow:
1. **Data Preparation**: Split the dataset into training and testing sets while maintaining class balance.
2. **Model Training**: Train the MLP classifier on the training set.
3. **Model Evaluation**: Evaluate the classifier's performance on the test set using metrics like accuracy.
4. **Visualization**: Visualize predictions and analyze misclassifications to understand model behavior.

This workflow mirrors the process used in the lecture notebook, but you will implement some of the functions yourself to deepen your understanding.


**Creating Train/Test Split.** We split using pandas so that each class contributes equally to train and test. `X_all` does not change; we simply split the indices stored in the dataframes.

**Do not change this split!** Otherwise the autograder will likely fail.


In [ ]:
meta_copy = meta.copy()
train_df = meta_copy.groupby('label').sample(frac=0.8, random_state=SEED)
test_df = meta_copy[~meta_copy.index.isin(train_df.index)].copy()
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

X_train = X_all[train_df["fmnist_idx"].to_numpy()]
X_test = X_all[test_df["fmnist_idx"].to_numpy()]
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"X_train shape: {X_train.shape}\t y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}\t y_test shape: {y_test.shape}")


### Problem 4a: Train MLP Classifier

In this task, we will train a Multi-Layer Perceptron (MLP) classifier to predict clothing categories from image data. The MLP is a type of neural network that is well-suited for classification tasks.

We have provided the code to define and fit the model, as well as extract and plot the loss curve. Your job is to normalize the data:
- Scale the pixel values of the images such that they have mean 0 and variance 1
- Create new variables `X_train_sc` and `X_test_sc` for the scaled training and testing data, respectively. Do not overwrite the original `X_train` and `X_test`.

**Notes:**
- The term "loss" refers to the error during training. Minimizing the loss is the goal of the training process.
- We set a specific `SEED` to ensure reproducability.
- You should use [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) from `sklearn` to scale the data. Use `fit_transform` only on the training data: `fit_transform` means that the scaler learns the scaling parameters from `X_train` and applies them to `X_train` in a single function call. Then use `transform` on the test data so it is scaled using those same parameters. For the rest of the homework, continue using `X_train_sc` and `X_test_sc`; you only need to fit and scale the data on the training set once, as long as you continue using the same scaled training data on the rest of the questions.

In [ ]:
# TODO: Train the model using the scaled training data and plot the loss curve (remember to normalize your data!)
# NOTE: Your model must be named `model`

if IS_GRADING_ENV:
    assert os.path.exists('classifier.joblib'), (
        "classifier.joblib is missing; the autograder does not retrain models"
    )
if load_saved_models and os.path.exists('classifier.joblib'):
    model = joblib.load('classifier.joblib')

    ...

    # Plot loss curve
    loss_df = pd.DataFrame({
        'epoch': range(1, len(model.loss_curve_) + 1),
        'loss': model.loss_curve_
    })
else:
    model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=100, tol=1e-3, random_state=SEED)

    ...

    model.fit(X=X_train_sc, y=y_train)

    # Plot loss curve
    loss_df = pd.DataFrame({
        'epoch': range(1, len(model.loss_curve_) + 1),
        'loss': model.loss_curve_
    })
if save_models:
    joblib.dump(model, 'classifier.joblib')

fig_q4a = loss_df.plot(x='epoch', y='loss', title="Training Error")
if not IS_GRADING_ENV:
    fig_q4a.show()


In [ ]:
grader.check("q4a")

### Problem 4b: Adding Predictions and Evaluation Metrics to DataFrames

**Task:** Modify both `train_df` and `test_df` by adding the following columns and compute train and test accuracy:

1. **`predicted_label`**: The predicted label for each image, as determined by the trained model.
2. **`correct`**: A boolean value indicating whether the predicted label matches the true label (`True` for correct predictions, `False` otherwise).
3. **`probs`**: The class probabilities for each image, represented as a list of size 10 (one probability per class).
4. **`confidence`**: The probability associated with the predicted label, representing the model's confidence in its prediction.

In [ ]:
# TODO: Add predicted_label, correct, probs, and confidence columns to train_df / test_df,
# then compute train_accuracy and test_accuracy.
train_df = train_df.copy()
test_df = test_df.copy()

...

print("--- Column Types ----")
for col in train_df.columns:
    val = train_df[col].iloc[0]
    print(f"{col}: {type(val)}")
print("-----------")

print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")


In [ ]:
grader.check("q4b")

### Problem 4c: Class Accuracy Analysis and Visualization

Analyze the model's performance for each class on both the training and testing datasets.

**Task:** Create a `class_accuracy` DataFrame
1. Group the `train_df` and `test_df` DataFrames by `label` (class).
2. Calculate the accuracy for each class as the proportion of correct predictions (`correct` column).
3. Add a `split` column to indicate whether the data is from the training or testing set.
4. Combine the results into a single DataFrame called `class_accuracy` with the following columns:
    - `split`: Indicates whether the data is from the training or testing set.
    - `label`: The class label.
    - `correct`: The accuracy for the class.

You will plot these accuracies in the next part.

**Hints**:
- Use `reset_index()` after grouping to convert the grouped data into a DataFrame.

For example, after a groupby:

    df.groupby(['A', 'B'])['C'].mean()

you get a Series with a multi-index:

```bash
    A      B       
    foo    x      0.92
           y      0.85
    bar    x      0.99
           y      0.97
    Name: C, dtype: float64
```

If you call `.reset_index()`, you get a `DataFrame` with columns:

```bash
       A    B     C
    0  foo  x  0.92
    1  foo  y  0.85
    2  bar  x  0.99
    3  bar  y  0.97
```

This makes it much easier to plot or further manipulate the data.


In [ ]:
# TODO: Calculate train and test accuracy per class. Store the result in `class_accuracy`.

...
print(class_accuracy)


In [ ]:
grader.check("q4ci")

<!-- BEGIN QUESTION -->

**Task:** Visualize class accuracy with a grouped bar chart.

1. Use the `class_accuracy` DataFrame to create a grouped bar chart.
2. The x-axis should represent the class labels (`label`), and the y-axis should represent the accuracy (`correct`).
3. Use different colors for the training and testing splits:
    - Training: Blue
    - Testing: Red
4. Add the actual accuracy values on top of the bars, rounded to two decimal places. To do this you can add `text_auto=True` to your `.plot` call. If you want to round these numbers to the nearest 2nd decimal, set `text_auto='.2f'`


In [ ]:
# TODO: Use `class_accuracy` to create a grouped bar chart of class accuracy for train and test. Assign the figure to `fig_q4cii`.

...
if not IS_GRADING_ENV:
    fig_q4cii.show()
save_figure(fig_q4cii, "q4cii")


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Problem 4d: Best and Worst Performing Classes

**Question:**  
- Identify the best and worst performing classes for train and test splits. If tied, list all classes with the same performance.  
- Do the best/worst performing classes match between splits?  
- Do train and test accuracies differ? Why?

Write your answer in the code cell below (in the `q4d_answer` string) and run that cell to save it as `written/q4d.txt`.


In [ ]:
# TODO: Write your answer in `q4d_answer`, then run this cell to save it.
q4d_answer = """
...
"""
save_written_answer(q4d_answer, "q4d")


<!-- END QUESTION -->

### Problem 4e: Create Confusion Matrix

An often easier way to understand model performance is with a confusion matrix, which shows how often predictions match the true labels and where errors occur. In this problem, we'll build a confusion matrix for the test set.

---

#### Refresher:
1. **Precision**: Measures the accuracy of positive predictions for a class.
  $$
  \text{Precision} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Positives}}
  $$

2. **Recall**: Measures the ability to identify all positive samples for a class.
  $$
  \text{Recall} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Negatives}}
  $$

---

**Task:** Hand-implement a confusion matrix for the test set and evaluate performance.
1. Use numpy operations to compute a 10x10 matrix where rows represent true labels and columns represent predicted labels.
2. Using your confusion matrix:
  - Compute overall test accuracy.
  - Calculate precision and recall for each class.

You will visualize the matrix in the next part.


In [ ]:
# Initialize confusion matrix with zeros
conf_matrix = np.zeros((len(class_names), len(class_names)), dtype=int)
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

# Fill the confusion matrix by counting predictions
...


In [ ]:
def metrics_from_confusion_matrix(conf_matrix, class_names):
    """
    Derive overall accuracy and per-class precision/recall from a confusion matrix.

    Parameters
    ----------
    conf_matrix : (C, C) ndarray
        Rows = true class, columns = predicted class.
    class_names : list[str]
        Class label names in the same order as the matrix axes.

    Returns
    -------
    accuracy_from_matrix : float
    per_class_metrics : list[dict]
        Each dict has keys: "class", "precision", "recall".
    """
    ...
    return accuracy_from_matrix, per_class_metrics


accuracy_from_matrix, per_class_metrics = metrics_from_confusion_matrix(conf_matrix, class_names)
print(f"\nAccuracy calculated from confusion matrix: {accuracy_from_matrix:.3f}")
print("\nPer-class metrics from confusion matrix:")
pd.DataFrame(per_class_metrics)


In [ ]:
grader.check("q4ei")

<!-- BEGIN QUESTION -->

**Task:** Visualize the confusion matrix.

Use a [heatmap](https://plotly.com/python/heatmaps/) to display `conf_matrix`. The y-axis should be the true label and the x-axis should be the predicted label.


In [ ]:
# TODO: Plot `conf_matrix` as a heatmap. Assign the figure to `fig_q4eii`.

...
if not IS_GRADING_ENV:
    fig_q4eii.show()
save_figure(fig_q4eii, "q4eii")


<!-- END QUESTION -->

### Problem 4f: Analyze Prediction Confidence

In this section, we will analyze the model's prediction confidence to better understand its behavior.

**Task:** Find **the image with the lowest confidence** by sorting the `confidence` column of `test_df`.


In [ ]:
# TODO: Find the image with the lowest confidence by sorting the `confidence` column of `test_df`.
...
print("Image with lowest confidence:")
print(least_confident[['label', 'predicted_label', 'confidence', 'correct']][:3])


In [ ]:
grader.check("q4fi")

<!-- BEGIN QUESTION -->

**Task:** Visualize a few of the least-confident examples with `show_images`, indexing into `X_test`.


In [ ]:
# TODO: Visualize a few of the least-confident examples with `show_images`, indexing into `X_test`. Assign the figure to `fig_q4fii`.
...
if not IS_GRADING_ENV:
    fig_q4fii.show()
save_figure(fig_q4fii, "q4fii")


<!-- END QUESTION -->

### Problem 4g: Investigating Class Confusion for "Trouser"

Now let's look at cases where the model is confidently incorrect.

**Task:** For the `Trouser` class, find the test-set images that are incorrectly classified as `Dress`. Store those rows in `test_df_trouser`, then sort them by confidence (highest first) into `high_conf_incorrect`.

You will plot them in the next part.


In [ ]:
# Find Trouser→Dress mistakes and sort by confidence (high→low).
# test_df_trouser = ...
# high_conf_incorrect = ...
...


In [ ]:
grader.check("q4gi")

<!-- BEGIN QUESTION -->

**Task:** Visualize the 10 images in `high_conf_incorrect` with the highest confidence using `show_images` and `X_test`.


In [ ]:
# TODO: Visualize the top high-confidence Trouser→Dress mistakes with `show_images`. Assign the figure to `fig_q4gii`.
...
if len(high_conf_incorrect) > 0:
    if not IS_GRADING_ENV:
        fig_q4gii.show()
    save_figure(fig_q4gii, "q4gii")


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Problem 4h: Reasons for High Confidence in the "Trouser" Class

**Task:** What are some potential reasons for the model to be so confident in its classifications of some of these examples?

Write your answer in the code cell below (in the `q4h_answer` string) and run that cell to save it as `written/q4h.txt`.


In [ ]:
# TODO: Write your answer in `q4h_answer`, then run this cell to save it.
q4h_answer = """
...
"""
save_written_answer(q4h_answer, "q4h")


<!-- END QUESTION -->

Now that we have become more familiar with the modeling process, let’s look at how we can augment our data and how these augmentations affect our classifier.

## Problem 5: Image Augmentation via Transformation Matrices

In this problem, you will explore how to implement image augmentations such as rotation, flipping, and scaling—using **matrix multiplication**. This carries on from Problem 5 in the written homework. The goal is to construct a transformation matrix $T$ such that, when multiplied by a flattened image vector, it produces the augmented image. Recall from the homework that normally $T$ expects images as column vectors. As we store our images as row vectors, we must first transpose our images and get transposed images as a result:

$$\text{augmented\_image}^\top = T \cdot \text{original\_image}^\top.$$

Alternatively, to get the un-transposed augmented image,

$$\text{augmented\_image} = \text{original\_image} \cdot T^\top \cdot$$

Each transformation matrix $T$ will be of size $N \times N$, where $N$ is the total number of pixels in the image (e.g., for a 28×28 image, $N=784$). Each row of $T$ defines how to compute the value of a single output pixel as a weighted sum of the input pixels.

---

### Why Use a Transformation Matrix?

Using a matrix for image transformations has several advantages:
1. **Efficiency**: Matrix multiplication is computationally efficient and can be optimized for hardware acceleration.
2. **Composability**: Multiple transformations (e.g., rotation followed by scaling) can be combined into a single matrix by multiplying their respective transformation matrices.
3. **Flexibility**: Any linear transformation, including interpolation, can be represented as a matrix.

---

### Example: Horizontal Flip Matrix

Let’s consider a simple example of flipping a 3×3 image horizontally. The flattened image is ordered row-wise:

Original indices:
$$\begin{bmatrix}
0 & 1 & 2 \\
3 & 4 & 5 \\
6 & 7 & 8
\end{bmatrix}$$

After a horizontal flip, the columns are reversed:
$$\begin{bmatrix}
2 & 1 & 0 \\
5 & 4 & 3 \\
8 & 7 & 6
\end{bmatrix}$$

The transformation matrix $T$ for this operation is a **permutation matrix** that swaps the columns for each row. For a 3×3 image, $T$ is a 9×9 matrix where each row has a single 1 in the position corresponding to the flipped pixel, and 0 elsewhere.

---

In this question, you will: 

1. **Understand Transformation Matrices**:
    - Learn how to construct transformation matrices for common operations like shifting, blurring, and rotating.

2. **Implement Augmentations**:
    - Write code to generate transformation matrices for the following operations:
      - **Shifting**: Move the image left, right, up, or down.
      - **Blurring**: Apply a smoothing effect by averaging neighboring pixels.
      - **Rotating**: Rotate the image by a specified angle.

3. **Combine Transformations**:
    - Experiment with combining multiple transformations into a single matrix and observe the results.


Whenever relevant, use `np.sin` and `np.cos` rather than `math.sin` and `math.cos`

Each method will consist of two steps:

1. **Create the Transformation Matrix**:  
    Construct a 784x784 transformation matrix that represents the desired image augmentation (e.g., rotation, flipping, scaling). Each row of the matrix determines how the value of a single output pixel is computed as a weighted sum of the input pixels.

2. **Apply the Transformation**:  
    Use the `apply_transformation` function (provided below) to apply the transformation matrix to your image. This function will handle the matrix multiplication and reshape the output back into the original image dimensions.

**Example: Vertical Flip**
To help you get started, we have implemented a simple vertical flip as an example. This transformation matrix swaps the rows of the image, flipping it vertically.

**Additional resources:** These slides from CS 180/280 (Computer Vision) might be helpful for conceptualizing and visualizing the transformations, especially the slides on rotations and bilinear interpolations! Note that not everything in the slides are relevant, so feel free to reference these as you see fit — [CS 180 lecture slides on image transformations and warping from Fa25](https://cal-cs180.github.io/fa25/Lectures/transformation_warping.pdf).

In [ ]:
def apply_transformation(image, T):
    # Input: A (N, 784) image vector and a (784, 784) transformation matrix
    # Output: A (N, 784) image vector
    transformed_flat = image @ T.T
    return transformed_flat.reshape(image.shape)

In [ ]:
def create_vertical_flip_matrix(height=28, width=28):
    """
    Returns a (height*width, height*width) matrix that vertically flips an image
    when applied to its flattened vector. Values are 0 or 1.
    """
    N = height * width  # Total number of pixels in the image
    T = np.zeros((N, N), dtype=int)  # Initialize the transformation matrix with zeros
    for i in range(height):  # Loop over each row
        for j in range(width):  # Loop over each column
            orig_idx = i * width + j  # Compute the flattened index for the original pixel
            flipped_i = height - 1 - i  # Compute the row index after vertical flip
            flipped_idx = flipped_i * width + j  # Compute the flattened index for the flipped pixel
            # Set the corresponding entry in the transformation matrix to 1
            # This means the pixel at (i, j) moves to (flipped_i, j)
            T[flipped_idx, orig_idx] = 1
    return T

def vertical_flip(image):
    T_flip = create_vertical_flip_matrix()
    return apply_transformation(image, T_flip)

In [ ]:
test_image = np.load("test_data/public/test_image.npy")

flipped_image = vertical_flip(test_image)
fig_q5_demo = show_images(np.stack([test_image, flipped_image]), labels=['Original', 'Flipped'], reshape=True)
if not IS_GRADING_ENV:
    fig_q5_demo.show()


### Problem 5a: Horizontal Flip

Now, let's implement a horizontal flip transformation using a matrix. A horizontal flip mirrors the image along its vertical axis. For example, the leftmost column becomes the rightmost column.

**Steps**:
1. **Understand the Transformation Matrix**:
    - The matrix `T` is `N x N` (where `N = height * width`).
    - Each row of `T` has a single `1` to indicate the new position of a pixel after the flip.

2. **Construct the Matrix**:
    - For each pixel `(i, j)`, compute its new position `(i, width - 1 - j)`.

3. **Apply the Transformation**:
    - Use the `apply_transformation` function to apply `T` to the flattened image.

**Hints**:
- Adjust the `flipped_j` and `flipped_idx` variables for the horizontal flip.
- Ensure the function returns a **flattened** image after applying the transformation.
- Fill any empty spaces in the transformed image with `0`


In [ ]:
def create_horizontal_flip_matrix(height=28, width=28):
    """
    Returns a (height*width, height*width) matrix that horizontally flips an image
    when applied to its flattened vector. Values are 0 or 1.
    """
    N = height * width
    T = np.zeros((N, N), dtype=int)
    ...
    return T

def horizontal_flip(image):
    T_flip = create_horizontal_flip_matrix()
    return apply_transformation(image, T_flip)

flipped_image = horizontal_flip(test_image)

fig_q5a = show_images(np.stack([test_image, flipped_image]), labels=['Original', 'Horizontal Flipped'], reshape=True)
if not IS_GRADING_ENV:
    fig_q5a.show()


In [ ]:
grader.check("q5a")

### Problem 5b: Image Shifting

**Task**: Implement a function to shift images by a specified number of pixels in any direction.

**Steps**:  
- Create a function that shifts an image by `dx` pixels horizontally and `dy` pixels vertically.  
- Fill empty spaces with 0s.  
- Handle cases where the shift moves parts of the image outside the boundaries.  
- Return the shifted image as a flattened array.

**Hint**:  
Think of copying pixels from a source region in the original image to a destination region in the final image. For example:  
- If `dx` is positive (shift right), the source x-range starts at 0 and ends at `28 - dx`.  
- If `dx` is negative (shift left), the source x-range starts at `-dx` and ends at 28.   
- If `dy` is positive (shift up), the source y-range starts at 0 and ends at `28 - dy`.  
- If `dy` is negative (shift down), the source y-range starts at `-dy` and ends at 28.  

Ensure the function returns a **flattened** image.

Fill any empty spaces in the transformed image with `0`

In [ ]:
def create_shift_matrix(dx, dy, height=28, width=28):
    """
    Create a transformation matrix for shifting an image by dx pixels horizontally and dy pixels vertically.

    Args:
        dx (int): Number of pixels to shift horizontally.
        dy (int): Number of pixels to shift vertically.
        height (int): Height of the image.
        width (int): Width of the image.

    Returns:
        np.ndarray: A (height*width, height*width) transformation matrix for shifting.
    """
    N = height * width
    T = np.zeros((N, N))
    ...
    return T


def shift_image(image, dx, dy):
    """
    Shift an image by dx pixels horizontally and dy pixels vertically.

    Args:
        image (np.ndarray): Flattened image array of shape (height*width,).
        dx (int): Number of pixels to shift horizontally.
        dy (int): Number of pixels to shift vertically.

    Returns:
        np.ndarray: Shifted image as a flattened array.
    """
    T = create_shift_matrix(dx, dy)
    return apply_transformation(image, T)

shifted_right_image = shift_image(test_image, 5, 0)
shifted_left_image = shift_image(test_image, -5, 0)
shifted_up_image = shift_image(test_image, 0, -5)
shifted_down_image = shift_image(test_image, 0, 5)

all_images = np.stack([test_image, shifted_up_image, shifted_down_image, shifted_right_image, shifted_left_image])
plot_labels = ['Original', 'Shifted Up', 'Shifted Down', 'Shifted Right', 'Shifted Left']
fig_q5b = show_images(all_images, labels=plot_labels, reshape=True)
if not IS_GRADING_ENV:
    fig_q5b.show()


In [ ]:
grader.check("q5b")

### Problem 5c: Image Blurring

**Task**  
Implement a blurring function using a transformation matrix that averages the values of neighboring pixels.

---

**What is blurring?**  
Blurring reduces the sharpness of an image by averaging each pixel with its neighbors, creating a smoother appearance.  
This is done with a **sliding square kernel** (window) that moves across the image.  
For each pixel, the kernel specifies which surrounding pixels contribute to the average.

---

**Key Concepts**
- **Kernel Size**  
  Controls how many neighbors are included in the average.
  - A **3×3 kernel** averages a pixel with its 8 immediate neighbors.
  - A **5×5 kernel** averages a pixel with its 24 neighbors.
- **Blurring Process**
  1. For each pixel, place a square kernel centered on that pixel.
  2. Collect all pixels that fall inside the kernel and inside the image.
  3. Compute the average of these valid pixels and assign it to the center pixel.

> *Edge handling:*  
> If the kernel extends beyond the image border, only the pixels that actually overlap the image are averaged.

---

**Example with a 4×4 image using a 3×3 kernel**

Original 4×4 image:

\begin{bmatrix}
10 & 20 & 30 & 40 \\
15 & 25 & 35 & 45 \\
50 & 60 & 70 & 80 \\
55 & 65 & 75 & 85
\end{bmatrix}


Consider the pixel with value 25 in row 2 col 2 [index (1, 1) in the matrix].  
Its 3×3 window contains:

\begin{bmatrix}
10 & 20 & 30 \\
15 & \textbf{25} & 35 \\
50 & 60 & 70
\end{bmatrix}

The blurred value for this position is the average of the numbers in this window (35).

For a corner pixel like (0, 0), the 3×3 window lies partly outside the image, so we average only the valid four neighbors:
\[
\frac{10 + 20 + 15 + 25}{4} = 17.5.
\]

Applying this process to every pixel produces a softened 4×4 image.

---

**Steps:**
1. Implement a function that, for each pixel, averages over a centered square window (kernel) of odd size (e.g., 3, 5, 7).  
   Handle edges by averaging only the valid neighbors.
2. Use a transformation matrix to apply this operation to the entire image.
3. Ensure the function works for any odd kernel size (e.g., 3x3, 5x5).
4. Return the blurred image as a flattened array.


Fill any empty spaces in the transformed image with `0`

In [ ]:
def create_blur_matrix(kernel_size=3, height=28, width=28):
    """
    Create a transformation matrix T that applies a uniform mean blur using a centered, odd-sized square sliding window.

    For each output pixel (i, j):
      1) Place a `kernel_size × kernel_size` window centered at (i, j).
      2) If the window is outside the image, then it will have fewer neighbors (only average the pixels that exist)

    Args:
        kernel_size (int): Size of the square kernel (must be odd).
        height (int): Height of the image.
        width (int): Width of the image.

    Returns:
        np.ndarray: A (height*width, height*width) transformation matrix for blurring.
    """
    N = height * width
    T = np.zeros((N, N))
    pad = kernel_size // 2

    ...
    return T

def blur_image(image, kernel_size=3):
    """
    Apply a blur transformation to a flattened image array or a batch of flattened images.

    Args:
        image (np.ndarray): Flattened image array of shape (height*width,) or batch of images (N, height*width).
        kernel_size (int): Size of the square kernel to use for blurring.

    Returns:
        np.ndarray: Blurred image(s) as a flattened array or batch of arrays.
    """
    T = create_blur_matrix(kernel_size)
    return apply_transformation(image, T)

blurred_1x1 = blur_image(test_image, kernel_size=1)
blurred_3x3 = blur_image(test_image, kernel_size=3)
blurred_5x5 = blur_image(test_image, kernel_size=5)

blurred_images = [test_image, blurred_1x1, blurred_3x3, blurred_5x5]
blurred_labels = ['Original', 'Blur 1x1', 'Blur 3x3', 'Blur 5x5']

fig_q5c = show_images(blurred_images, labels=blurred_labels, reshape=True)
if not IS_GRADING_ENV:
    fig_q5c.show()


In [ ]:
grader.check("q5c")

### Problem 5d: Image Rotation

**Task**: Implement a function to rotate an image by a given angle `theta` (in degrees).

**Steps**:
1. **Create the Rotation Matrix**:
    - Write a function `create_rotation_matrix(theta)` that generates a transformation matrix to rotate a flattened image by `theta` degrees.
    - Convert `theta` from degrees to radians using `np.deg2rad(theta)` before applying trigonometric functions.
    - Ensure the center of rotation is the center of the image.

2. **Apply the Transformation**:
    - The output should be a transformation matrix of shape `(height*width, height*width)`.
    - When this matrix is multiplied by the flattened image, it should produce the rotated image (also flattened).

**Hint**:
* Use trigonometric functions (`sin`, `cos`) to calculate the new positions of pixels after rotation. Use `np.sin` and `np.cos` instead of `math.sin` and `math.cos`.

In [ ]:
def create_rotation_matrix(theta, height=28, width=28):
    """
    Create a transformation matrix for rotating an image by theta degrees.

    Args:
        theta (float): Angle of rotation in degrees.
        height (int): Height of the image.
        width (int): Width of the image.

    Returns:
        np.ndarray: A (height*width, height*width) transformation matrix for rotating.
    """
    # Convert theta from degrees to radians
    theta = np.deg2rad(theta)
    N = height * width
    T = np.zeros((N, N))
    center_i = (height - 1) / 2.0
    center_j = (width - 1) / 2.0
    ...
    return T


def rotate_image(image, theta):
    """
    Apply a rotation transformation to a flattened image array or a batch of flattened images.

    Args:
        image (np.ndarray): Flattened image array of shape (height*width,) or batch of images (N, height*width).
        theta (float): Angle of rotation in degrees.

    Returns:
        np.ndarray: Rotated image(s) as a flattened array or batch of arrays.
    """
    T = create_rotation_matrix(theta)
    return apply_transformation(image, T)

# rotate with matrix
rotated_45 = rotate_image(test_image, 45) 
rotated_90 = rotate_image(test_image, 90)
rotated_200 = rotate_image(test_image, 200)
rotated_270 = rotate_image(test_image, 270)

# visualize original and 4 augmentations in plotly image grid
all_images = np.stack([test_image, rotated_45, rotated_90, rotated_200, rotated_270])
plot_labels = ['Original', 'Rotated (45°)', 'Rotated (90°)', 'Rotated (200°)', 'Rotated (270°)']
fig_q5d = show_images(all_images, labels=plot_labels, reshape=True)
if not IS_GRADING_ENV:
    fig_q5d.show()


In [ ]:
grader.check("q5d")

**Notice something?** For some rotations, we are left with holes in the image. 

### Understanding Gaps in Rotated Images

When rotating an image, you may notice white spaces (gaps) in the output. These gaps occur due to the way nearest-neighbor interpolation works. Let’s explore this using a simple $3 \times 3$ image.

---

**Original Image Grid**

The pixel coordinates are:

$$
\begin{bmatrix}
(0,0) & (0,1) & (0,2) \\
(1,0) & (1,1) & (1,2) \\
(2,0) & (2,1) & (2,2)
\end{bmatrix}
$$

The center of the image is at $(1,1)$.

---

**Rotation by $45^\circ$**

1. **Translate the center to the origin**  
   For pixel $(0,0)$:

   $$
   \begin{bmatrix} x \\ y \end{bmatrix}
   =
   \begin{bmatrix} 0 \\ 0 \end{bmatrix}
   -
   \begin{bmatrix} 1 \\ 1 \end{bmatrix}
   =
   \begin{bmatrix} -1 \\ -1 \end{bmatrix}
   $$

2. **Apply the rotation matrix**  
   The rotation matrix for $45^\circ$ is:

   $$
   R(45^\circ) =
   \tfrac{1}{\sqrt{2}}
   \begin{bmatrix}
   1 & -1 \\
   1 & 1
   \end{bmatrix}
   $$

   Applying the rotation:

   $$
   \begin{bmatrix} x' \\ y' \end{bmatrix}
   =
   R(45^\circ)
   \begin{bmatrix} -1 \\ -1 \end{bmatrix}
   =
   \tfrac{1}{\sqrt{2}}
   \begin{bmatrix} (-1) - (-1) \\ (-1) + (-1) \end{bmatrix}
   =
   \begin{bmatrix} 0 \\ -\sqrt{2} \end{bmatrix}
   \approx
   \begin{bmatrix} 0 \\ -1.4142 \end{bmatrix}
   $$

3. **Translate back to the original center**

   $$
   \begin{bmatrix} \text{new}_x \\ \text{new}_y \end{bmatrix}
   =
   \begin{bmatrix} 0 \\ -1.4142 \end{bmatrix}
   +
   \begin{bmatrix} 1 \\ 1 \end{bmatrix}
   \approx
   \begin{bmatrix} 1 \\ -0.4142 \end{bmatrix}
   $$

---

**Nearest-Neighbor Assignment**

To map the rotated pixel back to the grid, we round to the nearest integers:

$$
\text{new row} = \operatorname{round}(-0.4142) = 0, 
\quad
\text{new column} = \operatorname{round}(1) = 1
$$

Thus, pixel $(0,0)$ maps to $(0,1)$ in the rotated image.

---

**Why Do Gaps Appear?**

When mapping all pixels:

- **Overlaps**: Multiple original pixels may round to the same target coordinates.  
- **Gaps**: Some target coordinates are never assigned, leaving empty pixels (white spaces).

The rounding step in nearest-neighbor interpolation is the primary cause of these overlaps and gaps in the rotated image.

### Problem 5e: Bilinear Interpolation for Image Rotation

**Task**: When rotating an image, gaps (white spaces) can appear due to nearest-neighbor assignment. To avoid these gaps, set each output pixel to a weighted average of the 4 nearest source pixels. This approach is called [bilinear interpolation](https://en.wikipedia.org/wiki/Bilinear_interpolation) and is common in image processing for producing smoother, gap-free results.

Steps:
1) For each output pixel:
   - Translate its coordinates so that the rotation center is at the origin.
   - Apply the inverse rotation (i.e., rotate backward by the desired angle).
   - Translate the coordinates back to the original image space to locate the corresponding source position.
2) Perform interpolation:
       - Find the four nearest source pixels surrounding this position (top-left, top-right, bottom-left, bottom-right).
       - Compute the fractional distances from the source position to these neighbors (horizontal and vertical offsets).
       - Compute a weighted average of the four neighbor values using these offsets (bilinear interpolation).
       - Treat source pixels that fall outside of the original image as `0`.
3) Assign the computed value to the output pixel.
4) Repeat for all pixels. 


This method uses *inverse mapping* (sampling from the original image) rather than forward mapping (mapping source pixels to output), which helps prevent gaps.

**Hint:**
* Use trigonometric functions (`sin`, `cos`) to calculate the new positions of pixels after rotation. Use `np.sin` and `np.cos` instead of `math.sin` and `math.cos`.


In [ ]:
def create_bilinear_rotation_matrix(theta, height=28, width=28):
    """
    Create a (height*width, height*width) matrix that applies bilinear interpolation
    for rotating a flattened image by theta degrees.
    Each row of the matrix gives the weights for the input pixels that contribute to each output pixel.

    Args:
        theta (float): Angle of rotation in degrees.
        height (int): Height of the image.
        width (int): Width of the image.

    Returns:
        np.ndarray: A (height*width, height*width) transformation matrix for rotating.
    """
    theta = np.deg2rad(theta)
    N = height * width
    T = np.zeros((N, N))
    center_i = (height - 1) / 2.0
    center_j = (width - 1) / 2.0
    ...
    return T


def rotate_image_bilinear(image, theta):
    """
    Rotate an image using bilinear interpolation.

    Args:
        image (np.ndarray): Flattened image array of shape (height*width,) or batch of images (N, height*width).
        theta (float): Angle of rotation in degrees.

    Returns:
        np.ndarray: Rotated image as a flattened array.
    """
    T = create_bilinear_rotation_matrix(theta)
    return apply_transformation(image, T)
    
# rotate with matrix
rotated = rotate_image(test_image, 45)
rotated_interpolated = rotate_image_bilinear(test_image, 45)

all_images = np.stack([test_image, rotated, rotated_interpolated])
plot_labels = ['Original',  'Rotated 45°', 'Rotated 45° (Bilinear)']
fig_q5e = show_images(all_images, labels=plot_labels, reshape=True)
if not IS_GRADING_ENV:
    fig_q5e.show()


In [ ]:
grader.check("q5e")

### Problem 5f: Composing Transformations

An advantage of transformation matrices is their composability: you can combine multiple transformations into a single matrix. This allows you to apply multiple transformations to an image with the same computational cost as applying just one.

**Task**:

1. **Compose Multiple Transformations**:
    Implement `compose_transforms(*Ts)`, which takes any number of 784x784 transformation matrices (e.g., shift, rotate, blur) and returns a single matrix that represents applying all transformations in sequence. The transformations should be applied in the order they are provided: the first matrix is applied first, followed by the second, and so on.

2. **Rotate and Blur**:
    Implement `rotate_then_blur(image, theta, kernel_size)`, which rotates an image by `theta` degrees (without bilinear interpolation) and then applies a blur with a kernel of size `kernel_size`. Use `compose_transforms` to combine the transformations and apply them to the image.

3. **Shift, Rotate, and Blur**:
    Implement `shift_then_rotate_then_blur(image, dx, dy, theta, kernel_size)`, which shifts an image by `(dx, dy)`, rotates it by `theta` degrees (without bilinear interpolation), and then applies a blur with a kernel of size `kernel_size`. Again, use `compose_transforms` to combine the transformations and apply them to the image.

In [ ]:
def compose_transforms(*Ts):
  """
  Compose linear image transforms (each 784x784).
  Inputs:
    Ts: list of transformation matrices
  Returns:
    T_total: composition of all input transformations
  """
  ...
  return T_total

def rotate_then_blur(image, theta, kernel_size):
  """
  Rotate an image by theta degrees (without bilinear interpolation) and then blur it with a kernel of size kernel_size.
  """
  ...

def shift_then_rotate_then_blur(image, dx, dy, theta, kernel_size):
  """
  Shift an image by (dx, dy), then rotate it by theta degrees (without bilinear interpolation), and then blur it with a kernel of size kernel_size.
  """
  ...

rotated_blurred_image = rotate_then_blur(test_image, 45, 3)
shifted_rotated_blurred_image = shift_then_rotate_then_blur(test_image, 1, -4, 200, 5)

all_images = np.stack([test_image, rotated_blurred_image, shifted_rotated_blurred_image])
plot_labels = ['Original', 'Rotated 45° and Blurred 2x2', 'Shifted 5, Rotated 45° and Blurred 2x2']
fig_q5f = show_images(all_images, labels=plot_labels, reshape=True)
if not IS_GRADING_ENV:
    fig_q5f.show()


In [ ]:
grader.check("q5f")

<!-- BEGIN QUESTION -->

### Problem 5g: Matrix Multiply Questions
 
1. Does the order in which you apply transformations matter? Why or why not?  
2. When can a transformation be undone (i.e., when can you multiply your augmented image by another transformation matrix to recover the original image)? What matrix would you multiply by to recover the original image?  
3. Which of the augmentations implemented above can be "undone"? For augmentations that can be undone but may lose information (e.g., parts of the image are cut off), explain the conditions under which this occurs.  
4. Which of these augmentations *cannot* be "undone" with another matrix multiplication? Why not?

Write your answer in the code cell below (in the `q5g_answer` string) and run that cell to save it as `written/q5g.txt`.


In [ ]:
# TODO: Write your answer in `q5g_answer`, then run this cell to save it.
q5g_answer = """
...
"""
save_written_answer(q5g_answer, "q5g")


<!-- END QUESTION -->

### Testing Augmentation on Classifier Performance

In this section, we will evaluate how our trained classifier performs on augmented versions of the test images. This will help us understand the robustness of the model to various transformations.

The goal is to analyze the impact of different augmentation techniques on the classifier's performance. Specifically, we will:

1. **Create Augmented Test Images**:
    - Use the image augmentation functions (e.g., rotation, flipping, shifting, blurring) to generate transformed versions of the test images.

2. **Evaluate the Classifier**:
    - Test the classifier on the augmented images.
    - Measure and compare the accuracy for each augmentation type.

3. **Visualize Results**:
    - Plot the performance metrics to identify which augmentations the classifier handles well and which ones degrade performance.

### Problem 5h: Augmenting Test Images

**Task**: Create augmented versions of the test images using the image augmentation functions we implemented earlier.

**Steps**:
1. Apply each augmentation technique to a sample of 100 test images. This should result in 1300 images (13 augmentations $\times$ 100 test images).
2. Store **metadata** in a DataFrame `aug_df` with columns `original_idx`, `fmnist_idx`, `augmentation`, and `label`. The `augmentation` column should describe the augmentation applied.
3. Store the corresponding augmented pixel arrays in a NumPy array `augmented_images` with shape `(1300, 784)`, aligned row-for-row with `aug_df`.


In [ ]:
# Test augmentation functions on a few examples
test_images = X_test
test_labels = test_df['label'].to_numpy()
test_fmnist_idx = test_df['fmnist_idx'].to_numpy()

shift_inputs = [(5, 0), (-5, 0), (0, 5), (0, -5)]
rotate_inputs = [45, 90, 200]
blur_inputs = [3, 5]
rotate_blur_inputs = [(45, 3), (90, 5)]
shift_rotate_blur_inputs = [((5, 0), 45, 3), ((-5, 0), 90, 5)]

augmented_data = []
augmented_images_list = []
# Randomly sample 100 datapoints from test_images
sample_idx = np.random.choice(len(test_images), 100, replace=False)
test_images_sample = test_images[sample_idx]
test_labels_sample = np.array(test_labels)[sample_idx]
test_fmnist_sample = test_fmnist_idx[sample_idx]

# TODO: Apply shift / blur / rotate (bilinear) / rotate-then-blur / shift-rotate-blur
# to every image in test_images_sample using the input lists above.
# Store metadata rows in `augmented_data` and flat pixels in `augmented_images_list`,
# then build `aug_df` and `augmented_images`.
...


In [ ]:
grader.check("q5hi")

<!-- BEGIN QUESTION -->

**Task:** Visualize all augmentations for one sample image (for example, rows where `original_idx == 0`) using `show_images`.


In [ ]:
# TODO: Visualize all augmentations for one sample image (e.g. original_idx == 0). Assign the figure to `fig_q5hii`.
...
if not IS_GRADING_ENV:
    fig_q5hii.show()
save_figure(fig_q5hii, "q5hii")


<!-- END QUESTION -->

### Problem 5i: Evaluating Classifier Performance on Augmented Data

**Task:** Evaluate the classifier's performance on the augmented test data and compare its accuracy across different types of augmentations. Create a `DataFrame` named `aug_performance` with the following columns:
  - `augmentation`: A string describing the applied augmentation (e.g., "shift_5_0", "rotate_90", "blur_2x2").
  - `accuracy`: The classifier's accuracy on the augmented data.
  - `type`: The augmentation type (e.g., blur, rotate, shift, rotate_blur, shift_rotate_blur, none).

**Hints:**
1. Use `augmented_images` (aligned with `aug_df`) as the feature matrix — scale with the same `scaler` from training, then predict.
2. Aggregate correctness by the `augmentation` column of `aug_df` to build `aug_performance`.


In [ ]:
# TODO: Score the classifier on augmented images and summarize accuracy by augmentation type
# in `aug_performance`, then plot it.
...
if not IS_GRADING_ENV:
    fig_q5i.show()


In [ ]:
grader.check("q5i")

<!-- BEGIN QUESTION -->

### Problem 5j: Analysis of Augmentation Techniques

Among the augmentation techniques, which performed the best and which performed the worst? Why do you think this is the case? Provide reasoning based on the nature of the augmentations and their impact on the model's ability to generalize.

Write your answer in the code cell below (in the `q5j_answer` string) and run that cell to save it as `written/q5j.txt`.


In [ ]:
# TODO: Write your answer in `q5j_answer`, then run this cell to save it.
q5j_answer = """
...
"""
save_written_answer(q5j_answer, "q5j")


<!-- END QUESTION -->

You will being doing a LOT of matrix multiplication this semester, so get comfortable with these operations—they are fundamental to many machine learning algorithms you'll encounter!

# Before you submit, ensure save_models is true

Also run the next two cells: they check that your trained model and the expected `written/` figures and answers exist locally. Missing written files print a warning.


In [ ]:
assert save_models and load_saved_models, "save_models and load_saved_models must be True"

assert os.path.exists('classifier.joblib'), "classifier.joblib should exist"

In [ ]:
warn_missing_written_files()


Now that we have gotten familiar with pandas, numpy, and the classic training loop let's look into how we can debug and improve classifiers!

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

The cell below will generate a zip file for you to submit. It will also include all completed responses to manually graded questions saved under `written/` and other required files. **Please save before exporting!**


In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True, files=['classifier.joblib', 'written'])